## Overlap with CHIPseq (integrate with nichenet grn first)

In [ ]:
gtf_file = r"F:\gdT_aim2\SCENIC\SCENIC_database\Homo_sapiens.GRCh38.113.chr.gtf.gz"
gtf_data = pd.read_csv(gtf_file, sep="\t", comment="#", header=None)

C:\Users\16220\AppData\Local\Temp\ipykernel_26032\175377675.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  gtf_data = pd.read_csv(gtf_file, sep="\t", comment="#", header=None)


In [ ]:
# Filter for 'gene' feature
genes = gtf_data[gtf_data[2] == "gene"]

# Extract necessary columns (e.g., chromosome, start, end, strand, gene name)
genes = genes[[0, 3, 4, 6, 8]]
genes.columns = ["chrom", "start", "end", "strand", "attributes"]

# Extract gene names
genes["gene_name"] = genes["attributes"].str.extract('gene_name "([^"]+)"')

In [ ]:
no_source_grn = pd.read_csv(r"F:\gdT_aim2\SCENIC\predicted_GRN_no_source.csv")

In [ ]:
for tf in set(no_source_grn['tf']):
    target_genes = no_source_grn['target'][no_source_grn['tf']==tf]
    tss_regions = genes[genes["gene_name"].isin(target_genes)]
    tss_regions["tss"] = tss_regions.apply(lambda x: x["start"] if x["strand"] == "+" else x["end"], axis=1)
    tss_regions["start"] = tss_regions["tss"] - 5000
    tss_regions["end"] = tss_regions["tss"] + 5000
    tss_regions['chrom'] = 'chr'+tss_regions['chrom'].astype(str).map(str)

    # Save as BED file for overlap analysis
    tss_regions[["chrom", "start", "end", "gene_name"]].to_csv("SCENIC_database/TF_target_genes_tss/no_source_tss_5kb_for_"+tf+".bed", sep="\t", index=False, header=False)

C:\Users\16220\AppData\Local\Temp\ipykernel_26032\3866615841.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tss_regions["tss"] = tss_regions.apply(lambda x: x["start"] if x["strand"] == "+" else x["end"], axis=1)
C:\Users\16220\AppData\Local\Temp\ipykernel_26032\3866615841.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tss_regions["start"] = tss_regions["tss"] - 5000
C:\Users\16220\AppData\Local\Temp\ipykernel_26032\3866615841.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of

it is time to go back to R (CHIPseq_SCENIC_verify.R)

then, after getting the CHIP verification

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
values = [0,1]
colors = [(227, 227, 227), (15, 247, 74)]
norm = plt.Normalize(min(values), max(values))
my_cmap = LinearSegmentedColormap.from_list(
    '', [(norm(value), tuple(np.array(color) / 255)) for value, color in zip(values, colors)])
verified = pd.read_csv('verified_target_percentage.csv')
verified_percent =(verified*100).round(2)
%matplotlib inline
plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams.update({'font.size': 12, 'font.weight': 'heavy','axes.linewidth':2})
kwargs = {'cmap': my_cmap}



plot_df = verified_percent.copy().T[1]
plot_df.name = 'No Source Target Verified Percentage'
clustergrid = sns.clustermap(plot_df,figsize = (3,8),linewidths=2, vmin=0,vmax=100,
            linecolor = 'k',dendrogram_ratio = 0.1, colors_ratio = 0.5,
            row_cluster = False,col_cluster = False,cbar_kws={'label': 'Color Bar','location':"left"},
            **kwargs)


axs = clustergrid.fig.get_axes()
for j in range(0,len(axs)): 
    axs[j].set_xlabel('')

ax = clustergrid.ax_heatmap

# Extract data values and annotate each cell
for i in range(plot_df.shape[0]):  # Iterate over rows
    for j in range(1):  # Since there's only one column
        value = plot_df.iloc[i] # Get value
        # print(value)
        ax.text(j+0.5, i+0.5, f'{value}', ha='center', va='center', color='black', fontsize=12, fontweight='bold')
# plt.show()
plt.savefig('vis/ligand_verified.png',dpi = 300,bbox_inches='tight')